In [ ]:
%pip install torch

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import random

import torch
from torch.utils.data import Dataset


SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Carregamento e inspeção inicial dos dados

# Diretório raiz do projeto
PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

# Caminhos principais
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"


FILE_NAME = "dataset_chrun_processed.csv"
file_path = PROCESSED_DIR / FILE_NAME

df = pd.read_csv(file_path)

print(f"Shape do dataset: {df.shape}")
df.head()

In [ ]:
# Pre-Processamento

from sklearn.preprocessing import StandardScaler
from typing import Optional


def normalize_int(dataframe : pd.DataFrame) -> pd.DataFrame:

    # Normaliza as colunas numéricas inteiras usando MinMaxScaler
    numeric_features = dataframe.select_dtypes(include=["float64"]).columns.tolist()

    for cat in numeric_features:
        dataframe[cat] = dataframe[cat].astype(int)

    return dataframe

def pre_processing_data(dataframe : pd.DataFrame) -> pd.DataFrame:

    # identificar colunas numéricas e categóricas

    categorical_features = dataframe.select_dtypes(include=["object"]).columns.tolist()

    numeric_features = dataframe.select_dtypes(include=["int64", "float64"]).columns.tolist()

    # One-df encoding das variáveis categóricas relevantes

    dataframe = pd.get_dummies(dataframe, columns=categorical_features, drop_first=True)

    # Aplica StandardScaler

    scaler = StandardScaler()

    dataframe[numeric_features] = scaler.fit_transform(dataframe[numeric_features])

    
    # Converte as variaveis que passaram pelo encoder em numeros

    dataframe = bool_to_int(dataframe)

    # Converte as variaveis float em int
    dataframe = normalize_int(dataframe)

    return dataframe

def bool_to_int(dataframe : pd.DataFrame, bool_features : Optional[list] = None) -> pd.DataFrame:

      # Transforma dados booleanos em numéricos

    if bool_features is None:
        bool_features = dataframe.select_dtypes(include=["bool"]).columns.tolist()


    for cat in bool_features:
        dataframe[cat] = dataframe[cat].astype(bool)
        dataframe[cat] = dataframe[cat].astype(int)


    return dataframe




In [ ]:
# Separando os dados entre treino e teste

from sklearn.model_selection import train_test_split

# Converte variaveis booleanas para inteiros
df =bool_to_int(df, ['Churn'])

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X = pre_processing_data(X)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)


In [ ]:
# PyTorch - Variaveis

BATCH_SIZE = 64 # Qtde de amosras por batch
SHUFFLE = True # Embaralhaos dados a cada epica

In [ ]:
y.info()

In [ ]:
# PyTorch - Dataset

class ChurnDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)  # Corrigido para FloatTensor
        self.y = torch.FloatTensor(y.values)  # Corrigido para FloatTensor


    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

# Criando os datasets de treino e teste

train_dataset = ChurnDataset(X_train, y_train)
test_dataset = ChurnDataset(X_test, y_test)

In [ ]:
train_dataset

In [ ]:
# PyTorch - Importando os dados do dataset

from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
# Dispositivo para treino

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
# PyTorch - Criando o Modelo
import torch.nn as nn


class MLP(nn.Module):

    def __init__(self, qtd_features : int):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(qtd_features, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

In [ ]:
# PyTorch - Arquitetura do Modelo

import torch.optim as optim


model = MLP(qtd_features=X_train.shape[1])
criterion = nn.MSELoss()

# ----------------------------
# 8. Treino + Early stopping
# ----------------------------
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

patience = 5
best_val_loss = float("inf")
counter = 0

for epoch in range(200):

    # treino
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    # validação
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Train: {loss.item():.4f} | Val: {val_loss.item():.4f}")

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model = model.state_dict()
    else:
        counter += 1

        if counter >= patience:
            print("\n⛔ Early stopping!")
            break


In [ ]:

 
for epoch in range(10):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_dataloader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
from torch.nn import MSELoss
 
model.eval()
test_loss = 0
criterion = MSELoss()
 
with torch.no_grad():
    for batch_X, batch_y in test_dataloader:
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        test_loss += loss.item() * batch_X.size(0)
 
average_test_loss = test_loss / len(test_dataloader.dataset)
print(f"Test Mean Squared Error: {average_test_loss:.4f}")